In [2]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, binomtest

In [5]:
# Pfad zu deiner per-sample CSV
INPUT_CSV = "outputs/results_eval_per_sample_metrics.csv"

# Output-Ordner für neue Tabellen
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Die drei XAI-Methoden in deiner CSV
METHODS = ["gradcam", "rollout", "drise"]

METHOD_LABELS = {
    "gradcam": "Grad-CAM",
    "rollout": "Attention Rollout",
    "drise": "D-RISE",
}

In [6]:
df = pd.read_csv(INPUT_CSV)

print("Shape:", df.shape)
df.head()

Shape: (13179, 11)


,sample_id,image_id,file_name,method,pg,ebpg,iou,num_gt_boxes,target_query,target_class,target_score
0,1,1,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.118972...,gradcam,1,0.033855,0.047957,1,0,0,0.998898
1,2,2,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.240312...,gradcam,1,0.034990,0.095238,1,0,0,0.998307
2,3,3,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.292005...,gradcam,0,0.023513,0.040564,1,0,0,0.998878
3,4,4,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.337126...,gradcam,0,0.029157,0.040201,1,0,0,0.998888
4,5,5,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.337814...,gradcam,0,0.025093,0.038664,1,0,0,0.998044


In [7]:
required_cols = {"sample_id", "image_id", "file_name", "method", "pg", "ebpg", "iou"}
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["method"] = df["method"].str.lower()

print(df["method"].value_counts())

method
gradcam    4393
rollout    4393
drise      4393
Name: count, dtype: int64


In [8]:
def infer_modality(file_name):
    name = str(file_name).lower()

    # CT: Dateien mit 16bit
    if "16bit" in name:
        return "CT"

    # PET/CT: Dateien mit 8bit
    if "8bit" in name:
        return "PET/CT"

    return None


def infer_cancer_type(file_name):
    base = Path(str(file_name)).name
    first = base[0].upper()

    if first in {"A", "B", "E", "G"}:
        return first

    return None


df["modality"] = df["file_name"].apply(infer_modality)
df["cancer_type"] = df["file_name"].apply(infer_cancer_type)

df[["file_name", "modality", "cancer_type"]].head(-10)

,file_name,modality,cancer_type
0,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.118972...,CT,A
1,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.240312...,CT,A
2,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.292005...,CT,A
3,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.337126...,CT,A
4,A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.337814...,CT,A
...,...,...,...
13164,G0056_1.3.6.1.4.1.14519.5.2.1.6655.2359.777353...,CT,G
13165,G0056_1.3.6.1.4.1.14519.5.2.1.6655.2359.805254...,PET/CT,G
13166,G0056_1.3.6.1.4.1.14519.5.2.1.6655.2359.822024...,PET/CT,G
13167,G0056_1.3.6.1.4.1.14519.5.2.1.6655.2359.833350...,CT,G


In [9]:
# Wichtig: drop_duplicates, weil jedes Bild dreimal vorkommt: gradcam, rollout, drise
image_level = df[["image_id", "file_name", "modality", "cancer_type"]].drop_duplicates()

print("Modalities:")
print(image_level["modality"].value_counts(dropna=False))

print("\nCancer types:")
print(image_level["cancer_type"].value_counts(dropna=False))

Modalities:
modality
CT        2876
PET/CT    1517
Name: count, dtype: int64

Cancer types:
cancer_type
A    2922
G     795
B     597
E      79
Name: count, dtype: int64


In [10]:
grouped_csv = OUTPUT_DIR / "xai_per_sample_metrics_with_groups.csv"
df.to_csv(grouped_csv, index=False)

print("Saved:", grouped_csv)

Saved: outputs\xai_per_sample_metrics_with_groups.csv


In [11]:
def summarize_one_group(dataframe, group_name, group_value):
    rows = []

    for method in METHODS:
        sub = dataframe[dataframe["method"] == method]

        rows.append({
            "group": group_name,
            "group_value": group_value,
            "method": METHOD_LABELS[method],
            "n": int(len(sub)),
            "pg": float(sub["pg"].mean()) if len(sub) else np.nan,
            "ebpg": float(sub["ebpg"].mean()) if len(sub) else np.nan,
            "iou": float(sub["iou"].mean()) if len(sub) else np.nan,
            "pg_percent": float(sub["pg"].mean() * 100) if len(sub) else np.nan,
            "ebpg_percent": float(sub["ebpg"].mean() * 100) if len(sub) else np.nan,
            "iou_percent": float(sub["iou"].mean() * 100) if len(sub) else np.nan,
        })

    return rows


summary_rows = []

# Total
summary_rows.extend(
    summarize_one_group(
        dataframe=df,
        group_name="overall",
        group_value="Total",
    )
)

# Modalities
for modality in ["CT", "PET/CT"]:
    sub = df[df["modality"] == modality]
    summary_rows.extend(
        summarize_one_group(
            dataframe=sub,
            group_name="modality",
            group_value=modality,
        )
    )

# Cancer types
for cancer_type in ["A", "B", "E", "G"]:
    sub = df[df["cancer_type"] == cancer_type]
    summary_rows.extend(
        summarize_one_group(
            dataframe=sub,
            group_name="cancer_type",
            group_value=cancer_type,
        )
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,group,group_value,method,n,pg,ebpg,iou,pg_percent,ebpg_percent,iou_percent
0,overall,Total,Grad-CAM,4393,0.213749,0.028534,0.049406,21.374915,2.853414,4.940639
1,overall,Total,Attention Rollout,4393,0.895743,0.714305,0.444747,89.574323,71.430500,44.474734
2,overall,Total,D-RISE,4393,0.892329,0.016272,0.234276,89.232870,1.627242,23.427613
3,modality,CT,Grad-CAM,2876,0.233658,0.031092,0.055574,23.365786,3.109249,5.557381
4,modality,CT,Attention Rollout,2876,0.886300,0.711683,0.437007,88.630042,71.168268,43.700745
5,modality,CT,D-RISE,2876,0.882476,0.017376,0.222670,88.247566,1.737640,22.267025
6,modality,PET/CT,Grad-CAM,1517,0.176005,0.023684,0.037714,17.600527,2.368390,3.771390
7,modality,PET/CT,Attention Rollout,1517,0.913645,0.719277,0.459421,91.364535,71.927654,45.942099
8,modality,PET/CT,D-RISE,1517,0.911009,0.014179,0.256279,91.100857,1.417944,25.627911
9,cancer_type,A,Grad-CAM,2922,0.203285,0.026861,0.048357,20.328542,2.686149,4.835733


In [12]:
group_order = ["Total", "CT", "PET/CT", "A", "B", "E", "G"]

for group_value in group_order:
    sub = summary_df[summary_df["group_value"] == group_value]

    print(f"\n{group_value}")
    print("-" * 90)

    for _, row in sub.iterrows():
        print(
            f"{row['method']:20s} "
            f"PG={row['pg_percent']:6.2f}% | "
            f"EBPG={row['ebpg_percent']:6.2f}% | "
            f"IoU={row['iou_percent']:6.2f}% | "
            f"n={int(row['n'])}"
        )


Total
------------------------------------------------------------------------------------------
Grad-CAM             PG= 21.37% | EBPG=  2.85% | IoU=  4.94% | n=4393
Attention Rollout    PG= 89.57% | EBPG= 71.43% | IoU= 44.47% | n=4393
D-RISE               PG= 89.23% | EBPG=  1.63% | IoU= 23.43% | n=4393

CT
------------------------------------------------------------------------------------------
Grad-CAM             PG= 23.37% | EBPG=  3.11% | IoU=  5.56% | n=2876
Attention Rollout    PG= 88.63% | EBPG= 71.17% | IoU= 43.70% | n=2876
D-RISE               PG= 88.25% | EBPG=  1.74% | IoU= 22.27% | n=2876

PET/CT
------------------------------------------------------------------------------------------
Grad-CAM             PG= 17.60% | EBPG=  2.37% | IoU=  3.77% | n=1517
Attention Rollout    PG= 91.36% | EBPG= 71.93% | IoU= 45.94% | n=1517
D-RISE               PG= 91.10% | EBPG=  1.42% | IoU= 25.63% | n=1517

A
---------------------------------------------------------------------------

In [13]:
summary_csv = OUTPUT_DIR / "xai_summary_table.csv"
summary_df.to_csv(summary_csv, index=False)

print("Saved:", summary_csv)

Saved: outputs\xai_summary_table.csv


In [14]:
def mcnemar_exact(x, y):
    """
    Exact McNemar test für gepaarte binäre Daten.

    x, y: Arrays mit 0/1

    b = Methode A hit, Methode B miss
    c = Methode A miss, Methode B hit
    """
    x = np.asarray(x).astype(int)
    y = np.asarray(y).astype(int)

    b = int(np.sum((x == 1) & (y == 0)))
    c = int(np.sum((x == 0) & (y == 1)))
    n_discordant = b + c

    if n_discordant == 0:
        p_value = 1.0
    else:
        p_value = float(
            binomtest(
                k=min(b, c),
                n=n_discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )

    return {
        "b_a1_b0": b,
        "c_a0_b1": c,
        "n_discordant": n_discordant,
        "p_value": p_value,
    }

In [15]:
def holm_bonferroni(p_values):
    """
    Holm-Bonferroni-Korrektur.
    Gibt adjustierte p-Werte in Original-Reihenfolge zurück.
    """
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=float)

    previous = 0.0

    for rank, idx in enumerate(order):
        adjusted_p = (m - rank) * p_values[idx]
        adjusted_p = max(adjusted_p, previous)
        adjusted[idx] = min(adjusted_p, 1.0)
        previous = adjusted[idx]

    return adjusted

In [16]:
stats_rows = []

for metric in ["pg", "ebpg", "iou"]:
    # wide format:
    # sample_id | gradcam | rollout | drise
    wide = df.pivot(index="sample_id", columns="method", values=metric)

    # Nur Samples behalten, bei denen alle Methoden vorhanden sind
    wide = wide[METHODS].dropna()

    for method_a, method_b in combinations(METHODS, 2):
        a = wide[method_a].to_numpy()
        b = wide[method_b].to_numpy()
        diff = a - b

        row = {
            "metric": metric,
            "method_a": METHOD_LABELS[method_a],
            "method_b": METHOD_LABELS[method_b],
            "n_pairs": int(len(wide)),
            "mean_a": float(np.mean(a)),
            "mean_b": float(np.mean(b)),
            "median_a": float(np.median(a)),
            "median_b": float(np.median(b)),
            "mean_difference_a_minus_b": float(np.mean(diff)),
            "median_difference_a_minus_b": float(np.median(diff)),
            "n_a_greater_b": int(np.sum(diff > 0)),
            "n_b_greater_a": int(np.sum(diff < 0)),
            "n_equal": int(np.sum(diff == 0)),
        }

        if metric == "pg":
            result = mcnemar_exact(a, b)

            row.update({
                "test": "Exact McNemar test",
                "statistic": np.nan,
                "p_value": result["p_value"],
                "mcnemar_b_a1_b0": result["b_a1_b0"],
                "mcnemar_c_a0_b1": result["c_a0_b1"],
                "mcnemar_n_discordant": result["n_discordant"],
            })

        else:
            if np.all(diff == 0):
                statistic = 0.0
                p_value = 1.0
            else:
                result = wilcoxon(
                    a,
                    b,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto",
                )

                statistic = float(result.statistic)
                p_value = float(result.pvalue)

            row.update({
                "test": "Two-sided Wilcoxon signed-rank test",
                "statistic": statistic,
                "p_value": p_value,
                "mcnemar_b_a1_b0": np.nan,
                "mcnemar_c_a0_b1": np.nan,
                "mcnemar_n_discordant": np.nan,
            })

        stats_rows.append(row)

stats_df = pd.DataFrame(stats_rows)

# Multiple-testing correction über alle 9 Tests
stats_df["p_value_holm"] = holm_bonferroni(stats_df["p_value"].to_numpy())
stats_df["significant_alpha_0.05_holm"] = stats_df["p_value_holm"] < 0.05

stats_df

,metric,method_a,method_b,n_pairs,mean_a,mean_b,median_a,median_b,mean_difference_a_minus_b,median_difference_a_minus_b,...,n_b_greater_a,n_equal,test,statistic,p_value,mcnemar_b_a1_b0,mcnemar_c_a0_b1,mcnemar_n_discordant,p_value_holm,significant_alpha_0.05_holm
0,pg,Grad-CAM,Attention Rollout,4393,0.213749,0.895743,0.000000,1.000000,-0.681994,-1.000000,...,3040,1309,Exact McNemar test,NaN,0.00000,44.0,3040.0,3084.0,0.00000,True
1,pg,Grad-CAM,D-RISE,4393,0.213749,0.892329,0.000000,1.000000,-0.678580,-1.000000,...,3053,1268,Exact McNemar test,NaN,0.00000,72.0,3053.0,3125.0,0.00000,True
2,pg,Attention Rollout,D-RISE,4393,0.895743,0.892329,1.000000,1.000000,0.003415,0.000000,...,193,3992,Exact McNemar test,NaN,0.48452,208.0,193.0,401.0,0.48452,False
3,ebpg,Grad-CAM,Attention Rollout,4393,0.028534,0.714305,0.018757,0.767895,-0.685771,-0.737136,...,4287,1,Two-sided Wilcoxon signed-rank test,7437.0,0.00000,NaN,NaN,NaN,0.00000,True
4,ebpg,Grad-CAM,D-RISE,4393,0.028534,0.016272,0.018757,0.012749,0.012262,0.006285,...,654,0,Two-sided Wilcoxon signed-rank test,696412.0,0.00000,NaN,NaN,NaN,0.00000,True
5,ebpg,Attention Rollout,D-RISE,4393,0.714305,0.016272,0.767895,0.012749,0.698033,0.751797,...,100,0,Two-sided Wilcoxon signed-rank test,6260.0,0.00000,NaN,NaN,NaN,0.00000,True
6,iou,Grad-CAM,Attention Rollout,4393,0.049406,0.444747,0.021825,0.475906,-0.395341,-0.440837,...,4059,41,Two-sided Wilcoxon signed-rank test,141415.0,0.00000,NaN,NaN,NaN,0.00000,True
7,iou,Grad-CAM,D-RISE,4393,0.049406,0.234276,0.021825,0.169171,-0.184870,-0.129607,...,3751,27,Two-sided Wilcoxon signed-rank test,770305.0,0.00000,NaN,NaN,NaN,0.00000,True
8,iou,Attention Rollout,D-RISE,4393,0.444747,0.234276,0.475906,0.169171,0.210471,0.211814,...,956,82,Two-sided Wilcoxon signed-rank test,1087446.0,0.00000,NaN,NaN,NaN,0.00000,True


In [17]:
display_cols = [
    "metric",
    "method_a",
    "method_b",
    "test",
    "n_pairs",
    "mean_a",
    "mean_b",
    "p_value",
    "p_value_holm",
    "significant_alpha_0.05_holm",
]

stats_df[display_cols]

,metric,method_a,method_b,test,n_pairs,mean_a,mean_b,p_value,p_value_holm,significant_alpha_0.05_holm
0,pg,Grad-CAM,Attention Rollout,Exact McNemar test,4393,0.213749,0.895743,0.00000,0.00000,True
1,pg,Grad-CAM,D-RISE,Exact McNemar test,4393,0.213749,0.892329,0.00000,0.00000,True
2,pg,Attention Rollout,D-RISE,Exact McNemar test,4393,0.895743,0.892329,0.48452,0.48452,False
3,ebpg,Grad-CAM,Attention Rollout,Two-sided Wilcoxon signed-rank test,4393,0.028534,0.714305,0.00000,0.00000,True
4,ebpg,Grad-CAM,D-RISE,Two-sided Wilcoxon signed-rank test,4393,0.028534,0.016272,0.00000,0.00000,True
5,ebpg,Attention Rollout,D-RISE,Two-sided Wilcoxon signed-rank test,4393,0.714305,0.016272,0.00000,0.00000,True
6,iou,Grad-CAM,Attention Rollout,Two-sided Wilcoxon signed-rank test,4393,0.049406,0.444747,0.00000,0.00000,True
7,iou,Grad-CAM,D-RISE,Two-sided Wilcoxon signed-rank test,4393,0.049406,0.234276,0.00000,0.00000,True
8,iou,Attention Rollout,D-RISE,Two-sided Wilcoxon signed-rank test,4393,0.444747,0.234276,0.00000,0.00000,True


In [18]:
stats_csv = OUTPUT_DIR / "xai_statistical_tests_total.csv"
stats_df.to_csv(stats_csv, index=False)

print("Saved:", stats_csv)

Saved: outputs\xai_statistical_tests_total.csv


In [19]:
for _, row in stats_df.iterrows():
    metric = row["metric"].upper()
    method_a = row["method_a"]
    method_b = row["method_b"]
    p = row["p_value"]
    p_holm = row["p_value_holm"]
    significant = row["significant_alpha_0.05_holm"]

    sig_text = "significant" if significant else "not significant"

    print(
        f"{metric}: {method_a} vs. {method_b} | "
        f"p={p:.4g}, Holm-p={p_holm:.4g} → {sig_text}"
    )

PG: Grad-CAM vs. Attention Rollout | p=0, Holm-p=0 → significant
PG: Grad-CAM vs. D-RISE | p=0, Holm-p=0 → significant
PG: Attention Rollout vs. D-RISE | p=0.4845, Holm-p=0.4845 → not significant
EBPG: Grad-CAM vs. Attention Rollout | p=0, Holm-p=0 → significant
EBPG: Grad-CAM vs. D-RISE | p=0, Holm-p=0 → significant
EBPG: Attention Rollout vs. D-RISE | p=0, Holm-p=0 → significant
IOU: Grad-CAM vs. Attention Rollout | p=0, Holm-p=0 → significant
IOU: Grad-CAM vs. D-RISE | p=0, Holm-p=0 → significant
IOU: Attention Rollout vs. D-RISE | p=0, Holm-p=0 → significant
